# Mimi 32-Codebook Dataset Builder

This notebook creates a real 32-codebook Mimi token dataset from an audio dataset on Hugging Face.

It does **actual codec encoding**, then decodes a few samples back to WAV so you can verify the tokens are real before converting the full dataset.

Output row shape:

```text
id: string
text: string
speaker_id: string
codes: int16[32][n_frames]
n_frames: int32
k_codebooks: int32
sampling_rate: int32
duration_sec: float32
source_dataset: string
source_split: string
```

Important: this cannot upgrade an existing 8-codebook token dataset into 32 codebooks. It needs source audio.

In [ ]:
# Optional install cell. Run once on a fresh VM, then restart kernel if needed.
%pip install -U "datasets[audio]>=2.20.0" "transformers>=4.56.0" "accelerate>=0.33.0" "huggingface_hub>=0.34.0" "soundfile>=0.12.1" "pyarrow>=15.0.0" tqdm

In [ ]:
from pathlib import Path
import json
import math
import os
import time
from typing import Any

import numpy as np
import torch
import soundfile as sf
from datasets import Audio, Dataset, DatasetDict, Features, Sequence, Value, load_dataset, load_dataset_builder
from huggingface_hub import login
from IPython.display import Audio as DisplayAudio, display
from tqdm.auto import tqdm
from transformers import AutoFeatureExtractor, MimiModel

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## Configuration

Set `DATASET_NAME`, `DATASET_CONFIG`, `SPLITS`, and column names for your source dataset.

Good smoke-test default:

```text
hf-internal-testing/librispeech_asr_dummy, clean, validation
```

For real LibriTTS-R/LibriTTS-style data, set the dataset and config that actually expose an audio column.

In [ ]:
# Source dataset.
# For mythicinfinity/libritts_r, use config "all" and SPLITS="all".
DATASET_NAME = "mythicinfinity/libritts_r"
DATASET_CONFIG = "all"  # set to None if the dataset has no config
SPLITS = "all"          # "all" or a list like ["train.clean.100", "dev.clean"]
VERIFY_SPLIT = None     # None = first selected split

# Column names in the source dataset.
AUDIO_COLUMN = "audio"
TEXT_COLUMN = "text"
SPEAKER_COLUMN = "speaker_id"  # set to None if absent
ID_COLUMN = "id"               # set to None if absent; otherwise generated from split + row index

# Mimi settings.
MIMI_MODEL_NAME = "kyutai/mimi"
NUM_CODEBOOKS = 32

# Local output.
OUTPUT_ROOT = Path("data/mimi_32cb")
VERIFY_DIR = OUTPUT_ROOT / "verify_wavs"
PARQUET_DIR = OUTPUT_ROOT / "parquet_shards"

# Smoke verification first. Keep this small.
VERIFY_SAMPLES = 5
BENCHMARK_SAMPLES = 20

# Full conversion settings.
# Set PROCESS_LIMIT = None for the full selected split(s) after verification looks good.
# Keep PROCESS_LIMIT small for the first end-to-end run.
PROCESS_LIMIT = 100
SHARD_SIZE = 500

# Hugging Face upload settings. Fill these only when ready to push.
PUSH_TO_HUB = False
HF_REPO_ID = "your-username/your-mimi-32cb-dataset"
HF_PRIVATE = True
HF_TOKEN = os.environ.get("HF_TOKEN")  # or paste token string here, but env var is cleaner

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
VERIFY_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

print("output:", OUTPUT_ROOT.resolve())

In [ ]:
# Inspect source dataset metadata before loading everything.
if DATASET_CONFIG is None:
    builder = load_dataset_builder(DATASET_NAME)
else:
    builder = load_dataset_builder(DATASET_NAME, DATASET_CONFIG)

available_splits = list(builder.info.splits.keys()) if builder.info.splits else []

if SPLITS == "all":
    SELECTED_SPLITS = available_splits
elif isinstance(SPLITS, str):
    SELECTED_SPLITS = [SPLITS]
else:
    SELECTED_SPLITS = list(SPLITS)

missing = [s for s in SELECTED_SPLITS if available_splits and s not in available_splits]
if missing:
    raise ValueError(f"Unknown split(s): {missing}. Available: {available_splits}")

ACTIVE_VERIFY_SPLIT = VERIFY_SPLIT or SELECTED_SPLITS[0]

print(builder)
print("features:", builder.info.features)
print("available splits:", available_splits)
print("selected splits:", SELECTED_SPLITS)
print("verify split:", ACTIVE_VERIFY_SPLIT)

In [ ]:
# Load one verification split and force audio to Mimi sampling rate.
feature_extractor = AutoFeatureExtractor.from_pretrained(MIMI_MODEL_NAME)
TARGET_SR = int(feature_extractor.sampling_rate)
print("Mimi sampling rate:", TARGET_SR)


def load_source_split(split_name: str):
    load_kwargs = {"path": DATASET_NAME, "split": split_name}
    if DATASET_CONFIG is not None:
        load_kwargs["name"] = DATASET_CONFIG
    ds = load_dataset(**load_kwargs)
    ds = ds.cast_column(AUDIO_COLUMN, Audio(sampling_rate=TARGET_SR))
    return ds


source_ds = load_source_split(ACTIVE_VERIFY_SPLIT)

print(source_ds)
print("columns:", source_ds.column_names)
print("first row keys:", source_ds[0].keys())

In [ ]:
# Load the real Mimi codec model.
device = "cuda" if torch.cuda.is_available() else "cpu"
mimi = MimiModel.from_pretrained(MIMI_MODEL_NAME).to(device).eval()

print("device:", device)
config_quantizers = getattr(mimi.config, "num_quantizers", None)
print("mimi num_quantizers config:", config_quantizers)
print("requested codebooks:", NUM_CODEBOOKS)
if config_quantizers is not None:
    assert NUM_CODEBOOKS <= int(config_quantizers), "Requested more codebooks than this Mimi config supports."

## Helper Functions

These helpers do the real work:

```text
source audio -> Mimi feature extractor -> Mimi encode(num_quantizers=32) -> codes[32, frames]
```

The decode helper uses the same codes and Mimi decoder to reconstruct audio for inspection.

In [ ]:
OUTPUT_FEATURES = Features({
    "id": Value("string"),
    "text": Value("string"),
    "speaker_id": Value("string"),
    "codes": Sequence(Sequence(Value("int16"))),
    "n_frames": Value("int32"),
    "k_codebooks": Value("int32"),
    "sampling_rate": Value("int32"),
    "duration_sec": Value("float32"),
    "source_dataset": Value("string"),
    "source_split": Value("string"),
})


def safe_split_name(split_name: str) -> str:
    return split_name.replace("/", "_").replace(".", "_").replace("-", "_")


def get_row_id(row: dict[str, Any], idx: int, split_name: str) -> str:
    if ID_COLUMN and ID_COLUMN in row and row[ID_COLUMN] is not None:
        return str(row[ID_COLUMN])
    return f"{safe_split_name(split_name)}_{idx:09d}"


def get_speaker(row: dict[str, Any]) -> str:
    if SPEAKER_COLUMN and SPEAKER_COLUMN in row and row[SPEAKER_COLUMN] is not None:
        return str(row[SPEAKER_COLUMN])
    return "unknown"


def get_text(row: dict[str, Any]) -> str:
    value = row.get(TEXT_COLUMN, "")
    return "" if value is None else str(value)


def mono_float32(audio_array: np.ndarray) -> np.ndarray:
    audio = np.asarray(audio_array, dtype=np.float32)
    if audio.ndim == 2:
        # Accept either [channels, samples] or [samples, channels].
        if audio.shape[0] <= 8:
            audio = audio.mean(axis=0)
        else:
            audio = audio.mean(axis=1)
    if audio.ndim != 1:
        raise ValueError(f"Expected mono/stereo audio, got shape {audio.shape}")
    if not np.isfinite(audio).all():
        raise ValueError("Audio contains NaN/Inf")
    return audio


@torch.no_grad()
def encode_audio_to_mimi_codes(audio_array: np.ndarray) -> torch.Tensor:
    audio = mono_float32(audio_array)
    inputs = feature_extractor(
        raw_audio=audio,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    padding_mask = inputs.get("padding_mask")
    outputs = mimi.encode(
        inputs["input_values"],
        padding_mask,
        num_quantizers=NUM_CODEBOOKS,
        return_dict=True,
    )
    codes = outputs.audio_codes.detach().cpu().to(torch.int16)
    if codes.ndim != 3 or codes.shape[0] != 1 or codes.shape[1] != NUM_CODEBOOKS:
        raise RuntimeError(f"Unexpected codes shape: {tuple(codes.shape)}")
    return codes[0]  # [32, frames]


@torch.no_grad()
def decode_mimi_codes_to_audio(codes_2d: torch.Tensor) -> np.ndarray:
    codes = codes_2d.to(torch.long).unsqueeze(0).to(device)  # [1, 32, frames]
    decoded = mimi.decode(codes, return_dict=False)
    audio = decoded[0] if isinstance(decoded, tuple) else decoded.audio_values
    audio = audio.detach().float().cpu()
    if audio.ndim == 3:
        audio = audio[0, 0]
    elif audio.ndim == 2:
        audio = audio[0]
    audio_np = audio.numpy()
    peak = float(np.max(np.abs(audio_np))) if audio_np.size else 0.0
    if peak > 1.0:
        audio_np = audio_np / peak
    return audio_np.astype(np.float32)


def build_output_row(row: dict[str, Any], idx: int, split_name: str, codes: torch.Tensor) -> dict[str, Any]:
    audio = mono_float32(row[AUDIO_COLUMN]["array"])
    return {
        "id": get_row_id(row, idx, split_name),
        "text": get_text(row),
        "speaker_id": get_speaker(row),
        "codes": codes.numpy().astype(np.int16).tolist(),
        "n_frames": int(codes.shape[1]),
        "k_codebooks": int(codes.shape[0]),
        "sampling_rate": TARGET_SR,
        "duration_sec": float(len(audio) / TARGET_SR),
        "source_dataset": DATASET_NAME,
        "source_split": split_name,
    }

## Real Encode/Decode Verification

Run this before full conversion.

It will:

```text
1. Take real audio from the HF dataset.
2. Encode with Mimi using 32 codebooks.
3. Decode those exact codes back to waveform.
4. Save original and decoded WAVs.
5. Display audio players.
```

If these decoded samples sound like plausible reconstructions, the codec path is real.

In [ ]:
verify_rows = []
for idx in tqdm(range(min(VERIFY_SAMPLES, len(source_ds))), desc=f"verify encode/decode {ACTIVE_VERIFY_SPLIT}"):
    row = source_ds[idx]
    source_audio = mono_float32(row[AUDIO_COLUMN]["array"])
    codes = encode_audio_to_mimi_codes(source_audio)
    out = build_output_row(row, idx, ACTIVE_VERIFY_SPLIT, codes)
    verify_rows.append(out)

    decoded_audio = decode_mimi_codes_to_audio(codes)
    safe_split = safe_split_name(ACTIVE_VERIFY_SPLIT)
    original_wav = VERIFY_DIR / f"{safe_split}_{idx:03d}_original.wav"
    decoded_wav = VERIFY_DIR / f"{safe_split}_{idx:03d}_decoded_mimi_32cb.wav"
    sf.write(original_wav, source_audio, TARGET_SR)
    sf.write(decoded_wav, decoded_audio, TARGET_SR)

    print("\n---")
    print("split:", ACTIVE_VERIFY_SPLIT)
    print("id:", out["id"])
    print("text:", out["text"][:220])
    print("codes shape:", tuple(codes.shape), "min/max:", int(codes.min()), int(codes.max()))
    print("duration original/decoded:", round(len(source_audio) / TARGET_SR, 3), round(len(decoded_audio) / TARGET_SR, 3))
    print("original:", original_wav)
    display(DisplayAudio(str(original_wav), rate=TARGET_SR))
    print("decoded from real Mimi codes:", decoded_wav)
    display(DisplayAudio(str(decoded_wav), rate=TARGET_SR))

verify_ds = Dataset.from_list(verify_rows, features=OUTPUT_FEATURES)
verify_ds

In [ ]:
# Sanity-check the created rows.
for ex in verify_ds:
    assert ex["k_codebooks"] == NUM_CODEBOOKS
    assert len(ex["codes"]) == NUM_CODEBOOKS
    assert all(0 <= token <= 2047 for cb in ex["codes"] for token in cb), "Mimi code outside expected 0..2047 range"
    assert ex["n_frames"] == len(ex["codes"][0])
print("verification rows look valid")

## Speed Benchmark And Runtime Estimate

Run this on your actual machine before full conversion. The estimate depends heavily on GPU, audio duration, and dataset storage speed.

In [ ]:
bench_count = min(BENCHMARK_SAMPLES, len(source_ds))
wall_start = time.time()
audio_seconds = 0.0
frames_total = 0

for idx in tqdm(range(bench_count), desc=f"benchmark encode {ACTIVE_VERIFY_SPLIT}"):
    row = source_ds[idx]
    audio = mono_float32(row[AUDIO_COLUMN]["array"])
    audio_seconds += len(audio) / TARGET_SR
    codes = encode_audio_to_mimi_codes(audio)
    frames_total += int(codes.shape[1])

wall_sec = time.time() - wall_start
speed = audio_seconds / max(wall_sec, 1e-6)
print(f"encoded audio seconds: {audio_seconds:.2f}")
print(f"wall seconds: {wall_sec:.2f}")
print(f"speed: {speed:.2f}x realtime")
print(f"frames encoded: {frames_total}")

if builder.info.splits and ACTIVE_VERIFY_SPLIT in builder.info.splits:
    n_rows = int(builder.info.splits[ACTIVE_VERIFY_SPLIT].num_examples)
else:
    n_rows = len(source_ds)

avg_audio_sec = audio_seconds / max(bench_count, 1)
est_audio_sec = avg_audio_sec * n_rows
est_wall_sec = est_audio_sec / max(speed, 1e-6)
print("estimated rows for verify split:", n_rows)
print(f"estimated source audio hours for verify split: {est_audio_sec / 3600:.2f}")
print(f"estimated conversion wall hours for verify split: {est_wall_sec / 3600:.2f}")

if builder.info.splits:
    total_rows = sum(int(builder.info.splits[s].num_examples) for s in SELECTED_SPLITS)
    total_audio_sec = avg_audio_sec * total_rows
    total_wall_sec = total_audio_sec / max(speed, 1e-6)
    print("estimated rows for selected splits:", total_rows)
    print(f"rough source audio hours for selected splits: {total_audio_sec / 3600:.2f}")
    print(f"rough conversion wall hours for selected splits: {total_wall_sec / 3600:.2f}")

## Convert To Parquet Shards

After the verification audio sounds right, set `PROCESS_LIMIT = None` for the full split and run this cell.

For a large dataset, keep shard size moderate so progress is not lost if the VM stops.

In [ ]:
def write_shard(rows: list[dict[str, Any]], split_name: str, shard_index: int) -> Path:
    ds = Dataset.from_list(rows, features=OUTPUT_FEATURES)
    safe_split = safe_split_name(split_name)
    path = PARQUET_DIR / f"{safe_split}_mimi32_{shard_index:05d}.parquet"
    ds.to_parquet(str(path))
    return path

# Clear old shards only if you want a fresh run.
# for split_name in SELECTED_SPLITS:
#     for p in PARQUET_DIR.glob(f"{safe_split_name(split_name)}_mimi32_*.parquet"):
#         p.unlink()

all_new_shards = []
start_time = time.time()

for split_name in SELECTED_SPLITS:
    split_ds = load_source_split(split_name)
    limit = len(split_ds) if PROCESS_LIMIT is None else min(PROCESS_LIMIT, len(split_ds))
    safe_split = safe_split_name(split_name)
    print("\n=== split", split_name, "rows", limit, "of", len(split_ds), "===")
    print("writing shards to:", PARQUET_DIR.resolve())

    buffer = []
    shard_index = len(list(PARQUET_DIR.glob(f"{safe_split}_mimi32_*.parquet")))

    for idx in tqdm(range(limit), desc=f"encode {split_name}"):
        row = split_ds[idx]
        try:
            codes = encode_audio_to_mimi_codes(row[AUDIO_COLUMN]["array"])
            buffer.append(build_output_row(row, idx, split_name, codes))
        except Exception as exc:
            print(f"failed split={split_name} idx={idx}: {type(exc).__name__}: {exc}")
            continue

        if len(buffer) >= SHARD_SIZE:
            path = write_shard(buffer, split_name, shard_index)
            all_new_shards.append(path)
            print("wrote", path)
            buffer = []
            shard_index += 1

    if buffer:
        path = write_shard(buffer, split_name, shard_index)
        all_new_shards.append(path)
        print("wrote", path)

print("done")
print("new shards:", [str(p) for p in all_new_shards])
print("wall minutes:", round((time.time() - start_time) / 60, 2))

In [ ]:
# Load the produced parquet shards as a DatasetDict and inspect.
data_files = {}
for split_name in SELECTED_SPLITS:
    safe_split = safe_split_name(split_name)
    files = sorted(str(p) for p in PARQUET_DIR.glob(f"{safe_split}_mimi32_*.parquet"))
    if files:
        data_files[split_name] = files

print("splits with parquet shards:", {k: len(v) for k, v in data_files.items()})
assert data_files, "No parquet shards found. Run the conversion cell first."

encoded_dd = load_dataset("parquet", data_files=data_files)
print(encoded_dd)

first_split = next(iter(encoded_dd.keys()))
encoded_ds = encoded_dd[first_split]
print("first loaded split:", first_split)
print(encoded_ds.features)
print(encoded_ds[0].keys())
print("first codes shape:", len(encoded_ds[0]["codes"]), len(encoded_ds[0]["codes"][0]))

## Decode Random Converted Samples

This decodes from the saved dataset rows, not from the source audio, to verify the stored codes are usable.

In [ ]:
CHECK_DECODE_SAMPLES = min(5, len(encoded_ds))
for j in range(CHECK_DECODE_SAMPLES):
    ex = encoded_ds[j]
    codes = torch.tensor(ex["codes"], dtype=torch.long)
    decoded_audio = decode_mimi_codes_to_audio(codes)
    safe_split = safe_split_name(ex["source_split"])
    out_wav = VERIFY_DIR / f"stored_{safe_split}_{j:03d}_decoded_mimi_32cb.wav"
    sf.write(out_wav, decoded_audio, TARGET_SR)
    print("\n---")
    print("split:", ex["source_split"])
    print("id:", ex["id"])
    print("text:", ex["text"][:220])
    print("codes shape:", (len(ex["codes"]), len(ex["codes"][0])))
    print("saved:", out_wav)
    display(DisplayAudio(str(out_wav), rate=TARGET_SR))

## Push To Hugging Face

Only run this after verification passes.

Set:

```python
PUSH_TO_HUB = True
HF_REPO_ID = "your-username/dataset-name"
HF_TOKEN = "hf_..."  # or set env var HF_TOKEN
```

In [ ]:
if PUSH_TO_HUB:
    assert HF_REPO_ID and "/" in HF_REPO_ID, "Set HF_REPO_ID like 'username/dataset-name'"
    if HF_TOKEN:
        login(token=HF_TOKEN)
    else:
        login()

    encoded_dd.push_to_hub(HF_REPO_ID, private=HF_PRIVATE)
    print("pushed:", HF_REPO_ID)
else:
    print("PUSH_TO_HUB is False. Set it to True when ready.")

## Notes On Full LibriTTS-R Runtime

Run the benchmark cell on your actual GPU and trust that estimate over any guess.

Rough planning numbers:

```text
100 hours of source audio:
  5x realtime  -> about 20 wall hours
  10x realtime -> about 10 wall hours
  20x realtime -> about 5 wall hours

585 hours of source audio:
  5x realtime  -> about 117 wall hours
  10x realtime -> about 58 wall hours
  20x realtime -> about 29 wall hours
```

The one-by-one notebook path is built for correctness and resumable shards. After the first verified run, a batched/scripted encoder can make full conversion faster.